### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [9]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-20b")
response = model.invoke("Why do parrots talk in 80 words?")
response

AIMessage(content='**Short answer:**  \nParrots don’t have a built‑in “80‑word” limit. The number 80 is just a convenient round figure that people sometimes use in jokes or puzzles; in reality a parrot’s “vocabulary” can be dozens, hundreds, or even thousands of distinct sounds, depending on how much it’s been exposed to and how much training it gets.\n\n---\n\n## 1.  The biology behind parrot “talk”\n\n| Feature | What it does | Why it matters |\n|---------|--------------|----------------|\n| **Syrinx** | The bird’s vocal organ, located at the base of the trachea | It’s highly flexible and can produce a wide range of tones, much like a human larynx. |\n| **Neural circuitry** | Parrots have a large portion of their brain devoted to vocal learning (the “song system”) | Allows them to imitate and modify sounds they hear. |\n| **Memory & association** | They remember sounds and the contexts in which they’re used | They’ll repeat a phrase when the associated situation or cue appears. |\n\n

In [10]:
from langchain.tools import tool
@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""
    return f"The weather in {location} is sunny with a high of 25°C."

model_with_tool = model.bind_tools([get_weather])

In [11]:
response = model_with_tool.invoke("What is the weather in New York?")
print(response)

for too_call in response.tool_calls:
    print(f"Tool Name: {too_call['name']}")
    print(f"Tool args: {too_call['args']}")

content='' additional_kwargs={'reasoning_content': 'User asks for weather in New York. We have a function get_weather. Use it.', 'tool_calls': [{'id': 'fc_f98562c5-a9b3-483f-b5ae-6653ae17f341', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 128, 'total_tokens': 171, 'completion_time': 0.0459402, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.007378116, 'prompt_tokens_details': None, 'queue_time': 0.191512079, 'total_time': 0.053318316}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e99e93f2ac', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a08046-171c-7181-8a9a-812a01052cc7-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'fc_f98562c5-a9b3-483f-b5ae-6653ae17f341', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metad

### Tool Execution Loops

In [12]:
messages = [{"role": "user", "content": "What is the weather in New York?"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tool.invoke(messages)
print(final_response.text)

The weather in New York is sunny with a high of 25 °C.


In [13]:
messages

[{'role': 'user', 'content': 'What is the weather in New York?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call get_weather with location "New York".', 'tool_calls': [{'id': 'fc_999e3f9e-23d5-439c-bb08-5607d0fc2cd7', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 128, 'total_tokens': 165, 'completion_time': 0.037901267, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.007223568, 'prompt_tokens_details': None, 'queue_time': 0.0312795, 'total_time': 0.045124835}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_7d448090ba', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a08049-0c29-7eb0-8b48-9f84867ccc7f-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'fc_999e3f9e-23d5-439c-bb08-5607d0fc2c